In [1]:
# =============================================================================
# Section 3 - Recommendation Status Prediction (Optimized for Speed)
# =============================================================================

import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split, RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
FA_TOKEN_PATTERN = r"[\u0600-\u06FF0-9A-Za-z]+"
LABELS = ["recommended", "no_idea", "not_recommended"]


In [2]:

# -----------------------------------------------------------------------------
# 1. Data Loading & Cleaning (Same as before)
# -----------------------------------------------------------------------------

REC_COLUMNS = [
    "comment_id", "product_id", "title_norm", "body_norm", "advantages_norm",
    "disadvantages_norm", "comment_text_norm", "recommendation_status",
    "recommendation_valid", "has_text", "rate_clean", "likes", "is_buyer"
]

rec_raw = pd.read_csv('D:/Quera_bootcamp/LLM/comments_clean.csv', usecols=REC_COLUMNS)

mask = (
    rec_raw["recommendation_valid"].fillna(False) &
    rec_raw["has_text"].fillna(False) &
    rec_raw["product_id"].notna()
)
rec_df = rec_raw.loc[mask].drop(columns=["recommendation_valid", "has_text"]).copy()
rec_df = rec_df.drop_duplicates(subset="comment_text_norm", keep="first")

# -----------------------------------------------------------------------------
# 2. Encode target labels
# -----------------------------------------------------------------------------

le = LabelEncoder()
rec_df['target_encoded'] = le.fit_transform(rec_df['recommendation_status'])
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
inv_label_mapping = {v: k for k, v in label_mapping.items()}


In [3]:

# -----------------------------------------------------------------------------
# 3. Stratified Sampling (Reduce to 30k per class for faster training)
# -----------------------------------------------------------------------------

MAX_PER_CLASS = 30_000  # Reduced from 60k to speed up

def stratified_cap(df, label_col, max_per_class, random_state):
    parts = [
        group.sample(n=min(len(group), max_per_class), random_state=random_state)
        for _, group in df.groupby(label_col, sort=False)
    ]
    return pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)

rec_sample = stratified_cap(rec_df, "target_encoded", MAX_PER_CLASS, RANDOM_STATE)

# -----------------------------------------------------------------------------
# 4. Train / Validation / Test Split (60/20/20 for stability)
# -----------------------------------------------------------------------------

train_df, temp_df = train_test_split(
    rec_sample, test_size=0.40,
    stratify=rec_sample["target_encoded"],
    random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df["target_encoded"],
    random_state=RANDOM_STATE
)

assert not (set(train_df["comment_text_norm"]) & set(val_df["comment_text_norm"]))
assert not (set(train_df["comment_text_norm"]) & set(test_df["comment_text_norm"]))
assert not (set(val_df["comment_text_norm"]) & set(test_df["comment_text_norm"]))


In [4]:

# -----------------------------------------------------------------------------
# 5. Numeric Features
# -----------------------------------------------------------------------------

NUMERIC_FEATURES = ['rate_clean', 'likes', 'is_buyer_num']

for df in [train_df, val_df, test_df]:
    df['is_buyer_num'] = df['is_buyer'].fillna(False).astype(int)
    df['rate_clean'] = df['rate_clean'].fillna(0)
    df['likes'] = df['likes'].fillna(0)


In [5]:

# -----------------------------------------------------------------------------
# 6. Text Vectorizer (Reduced max_features to 50k for speed)
# -----------------------------------------------------------------------------

PERSIAN_STOPWORDS = [
    'و', 'در', 'به', 'از', 'که', 'این', 'با', 'را', 'برای', 'رو', 'هم',
    'یک', 'ها', 'است', 'نیز', 'شد', 'شود', 'می', 'خواهد', 'بر', 'آن',
    'همان', 'تا', 'کرد', 'کردن', 'گفت', 'داشت', 'دارد', 'داشته', 'بود',
    'بوده', 'باش', 'باشد', 'باشم', 'باشی', 'باشید', 'باشند', 'باشیم',
    'نمود', 'نمودن', 'ساخت', 'ساختن', 'زیر', 'روی', 'پشت', 'جلوی',
    'کنار', 'بین', 'میان', 'طریق', 'بعد', 'قبل', 'حتی', 'مثل', 'مانند',
    'اما', 'اگر', 'هر', 'همه', 'چون', 'چرا', 'چطور', 'چگونه', 'چه',
    'کی', 'کجا', 'کدام', 'چند', 'بسیار', 'خیلی', 'کمی', 'بیشتر', 'کمتر'
]

def build_vectorizer(max_features=50_000):
    return TfidfVectorizer(
        token_pattern=FA_TOKEN_PATTERN,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.8,
        sublinear_tf=True,
        max_features=max_features,
        stop_words=PERSIAN_STOPWORDS
    )


In [6]:

# -----------------------------------------------------------------------------
# 7. Pipeline Builder
# -----------------------------------------------------------------------------

def create_pipeline(classifier, use_smote=False, smote_kwargs=None, include_numeric=True):
    transformers = [('text', build_vectorizer(), 'comment_text_norm')]
    if include_numeric:
        transformers.append(('num', StandardScaler(), NUMERIC_FEATURES))
    preprocessor = ColumnTransformer(transformers)

    if use_smote:
        pipeline = ImbPipeline([
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE, **(smote_kwargs or {}))),
            ('clf', classifier)
        ])
    else:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('clf', classifier)
        ])
    return pipeline

def prepare_for_pipeline(df):
    X = df[['comment_text_norm'] + NUMERIC_FEATURES].copy()
    y = df['target_encoded']
    return X, y

X_train, y_train = prepare_for_pipeline(train_df)
X_val, y_val = prepare_for_pipeline(val_df)
X_test, y_test = prepare_for_pipeline(test_df)


In [7]:

# -----------------------------------------------------------------------------
# 8. Baseline Evaluation (Fast models only)
# -----------------------------------------------------------------------------

BASE_MODELS = {
    'majority_baseline': DummyClassifier(strategy='most_frequent'),
    'logistic_regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'linear_svm': LinearSVC(
        class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE, dual='auto'
    )
}

baseline_results = []
for name, clf in BASE_MODELS.items():
    pipe = create_pipeline(clf, use_smote=False, include_numeric=True)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    baseline_results.append({'model': name, 'val_macro_f1': macro_f1})

baseline_df = pd.DataFrame(baseline_results).sort_values('val_macro_f1', ascending=False)


In [8]:

# -----------------------------------------------------------------------------
# 9. Fast Hyperparameter Tuning with RandomizedSearchCV (Logistic Regression only)
# -----------------------------------------------------------------------------

print("Starting RandomizedSearchCV (will take ~5-10 minutes)...")

param_dist = {
    'clf__C': [0.1, 0.5, 1.0, 2.0, 5.0],
    'clf__solver': ['liblinear', 'saga'],
    'clf__penalty': ['l1', 'l2']
}

base_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
pipe = create_pipeline(base_lr, use_smote=True, smote_kwargs={'k_neighbors': 3}, include_numeric=True)

random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=6,               # Only 6 combinations instead of full grid
    cv=2,                   # 2-fold CV to speed up
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
random_search.fit(X_train, y_train)

best_pipeline = random_search.best_estimator_
best_params = random_search.best_params_
print("Best parameters:", best_params)
print("Best CV score:", random_search.best_score_)


Starting RandomizedSearchCV (will take ~5-10 minutes)...
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Best parameters: {'clf__solver': 'saga', 'clf__penalty': 'l1', 'clf__C': 0.5}
Best CV score: 0.7641839933518435


In [9]:

# -----------------------------------------------------------------------------
# 10. Final Evaluation on Test Set
# -----------------------------------------------------------------------------

y_pred_test = best_pipeline.predict(X_test)
test_macro_f1 = f1_score(y_test, y_pred_test, average='macro')
test_accuracy = accuracy_score(y_test, y_pred_test)

# Decode for display
y_test_decoded = [inv_label_mapping[i] for i in y_test]
y_pred_decoded = [inv_label_mapping[i] for i in y_pred_test]

cm = confusion_matrix(y_test_decoded, y_pred_decoded, labels=LABELS)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

test_view = test_df.copy()
test_view['predicted'] = [inv_label_mapping[i] for i in y_pred_test]
mistakes = test_view[test_view['recommendation_status'] != test_view['predicted']]

print("\nClassification Report on Test Set:")
print(classification_report(y_test_decoded, y_pred_decoded, digits=3))



Classification Report on Test Set:
                 precision    recall  f1-score   support

        no_idea      0.687     0.683     0.685      6000
not_recommended      0.795     0.829     0.811      6000
    recommended      0.818     0.788     0.803      6000

       accuracy                          0.766     18000
      macro avg      0.767     0.767     0.766     18000
   weighted avg      0.767     0.766     0.766     18000



In [11]:
# -----------------------------------------------------------------------------
# 11. Product-Grouped Validation (No product overlap)
# -----------------------------------------------------------------------------

print("\nRunning product-grouped validation...")

# Ensure rec_sample has numeric features (needed for prepare_for_pipeline)
rec_sample['is_buyer_num'] = rec_sample['is_buyer'].fillna(False).astype(int)
rec_sample['rate_clean'] = rec_sample['rate_clean'].fillna(0)
rec_sample['likes'] = rec_sample['likes'].fillna(0)

gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
grp_train_idx, grp_temp_idx = next(
    gss_outer.split(rec_sample, groups=rec_sample['product_id'])
)
grp_train_df = rec_sample.iloc[grp_train_idx]
grp_temp_df = rec_sample.iloc[grp_temp_idx]

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
grp_val_idx, grp_test_idx = next(
    gss_inner.split(grp_temp_df, groups=grp_temp_df['product_id'])
)
grp_test_df = grp_temp_df.iloc[grp_test_idx]

assert not (set(grp_train_df['product_id']) & set(grp_test_df['product_id']))

X_grp_train, y_grp_train = prepare_for_pipeline(grp_train_df)
X_grp_test, y_grp_test = prepare_for_pipeline(grp_test_df)

grp_pipeline = create_pipeline(
    best_pipeline.named_steps['clf'],
    use_smote=False,
    include_numeric=True
)
grp_pipeline.fit(X_grp_train, y_grp_train)
grp_test_macro_f1 = f1_score(
    y_grp_test, grp_pipeline.predict(X_grp_test), average='macro'
)


Running product-grouped validation...


In [12]:

# -----------------------------------------------------------------------------
# 12. Ablation Study (Simplified)
# -----------------------------------------------------------------------------

ablation_results = []
configs = [
    ('text_only', ['comment_text_norm']),
    ('text_and_numeric', ['comment_text_norm'] + NUMERIC_FEATURES),
]

for name, feature_cols in configs:
    if 'comment_text_norm' in feature_cols:
        vec = build_vectorizer()
        X_tr = vec.fit_transform(X_train['comment_text_norm'])
        X_v = vec.transform(X_val['comment_text_norm'])
        if name == 'text_only':
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, C=1.0)
            clf.fit(X_tr, y_train)
            y_pred = clf.predict(X_v)
        else:
            scaler = StandardScaler()
            X_tr_num = scaler.fit_transform(X_train[NUMERIC_FEATURES])
            X_v_num = scaler.transform(X_val[NUMERIC_FEATURES])
            X_tr_comb = hstack([X_tr, X_tr_num])
            X_v_comb = hstack([X_v, X_v_num])
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, C=1.0)
            clf.fit(X_tr_comb, y_train)
            y_pred = clf.predict(X_v_comb)
        macro_f1 = f1_score(y_val, y_pred, average='macro')
        ablation_results.append({'feature_set': name, 'val_macro_f1': macro_f1})

ablation_df = pd.DataFrame(ablation_results).sort_values('val_macro_f1', ascending=False)


In [14]:
# -----------------------------------------------------------------------------
# 13. Save Model
# -----------------------------------------------------------------------------

# تعریف PROCESSED_DIR در صورت عدم وجود
from pathlib import Path

# اگر در جای دیگری تعریف نشده، اینجا تعریف کن
if 'PROCESSED_DIR' not in globals():
    BASE_DIR = Path("D:/Quera_bootcamp/LLM")   # مسیر پروژه خودت را تنظیم کن
    PROCESSED_DIR = BASE_DIR / "data_processed"
    PROCESSED_DIR.mkdir(exist_ok=True)

MODEL_PATH = PROCESSED_DIR / "recommendation_model_v2.pkl"
joblib.dump({'pipeline': best_pipeline, 'label_encoder': le}, MODEL_PATH)

# -----------------------------------------------------------------------------
# 14. Summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print("Best model: Logistic Regression with SMOTE")
print("Best params:", best_params)
print("Test accuracy:", round(test_accuracy, 4))
print("Test macro F1:", round(test_macro_f1, 4))

# نمایش product-grouped macro F1 در صورت وجود
if 'grp_test_macro_f1' in globals():
    print("Product-grouped macro F1:", round(grp_test_macro_f1, 4))
else:
    print("Product-grouped macro F1: (not computed)")

print("Model saved to:", MODEL_PATH)

# نمایش ablation study در صورت وجود
if 'ablation_df' in globals():
    print("Ablation study:")
    print(ablation_df.to_string(index=False))
else:
    print("Ablation study: (not computed)")
print("=" * 50)


SUMMARY
Best model: Logistic Regression with SMOTE
Best params: {'clf__solver': 'saga', 'clf__penalty': 'l1', 'clf__C': 0.5}
Test accuracy: 0.7665
Test macro F1: 0.7663
Product-grouped macro F1: 0.7695
Model saved to: D:\Quera_bootcamp\LLM\data_processed\recommendation_model_v2.pkl
Ablation study:
     feature_set  val_macro_f1
text_and_numeric      0.761727
       text_only      0.723378
